<a href="https://colab.research.google.com/github/shrinikethm/nba_modeling_injury_prevention/blob/main/NBA_Injury_Prediction_WithDashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Imports

This cell imports necessary libraries for data manipulation, machine learning, and visualization.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns


### Upload Data

This cell handles uploading the CSV file containing player injury details to the Colab environment.

In [ ]:
from google.colab import files
uploaded = files.upload()

### Load Data and Initial Exploration

This cell loads the uploaded CSV data into a Pandas DataFrame and displays the first few rows and summary information to understand its structure and data types.

In [ ]:
df = pd.read_csv('player_injury_details_nov25_new.csv')

df.head()
df.info()


### Data Preprocessing for Logistic Regression (Initial)

This cell prepares the data by dropping irrelevant columns, converting 'Draft' related columns to numeric, encoding categorical variables, and handling any remaining missing values for the initial Logistic Regression model.

In [ ]:
# Drop identifiers and College for simplicity
drop_cols = ['PlayerFull', 'PlayerID', 'TeamName', 'SeasonID', 'College']
X = df.drop(columns=drop_cols + ['Injury', 'InjuryCount'])
y = df['Injury']

# Convert Draft columns to numeric if possible
X['DraftYear'] = pd.to_numeric(X['DraftYear'], errors='coerce').fillna(0)
X['DraftRound'] = pd.to_numeric(X['DraftRound'], errors='coerce').fillna(0)
X['DraftPick'] = pd.to_numeric(X['DraftPick'], errors='coerce').fillna(0)

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Fill any remaining missing values
X = X.fillna(0)

### Train and Evaluate Initial Logistic Regression Model

This cell splits the preprocessed data into training and testing sets, scales the features, trains a Logistic Regression model, and then evaluates its performance using accuracy, a confusion matrix, and a classification report.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# Predict and evaluate
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Classification report
print(classification_report(y_test, y_pred))

### Visualize Logistic Regression Coefficients (Initial Model)

This cell extracts and displays the coefficients of the initial Logistic Regression model, indicating the importance and direction of influence for each feature on injury prediction. It also visualizes the top 10 contributing factors.

In [ ]:
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', key=abs, ascending=False)

print(coefficients.head(10))

# Optional: visualize top 10 factors
plt.figure(figsize=(10,6))
sns.barplot(x='Coefficient', y='Feature', data=coefficients.head(10))
plt.title('Top 10 Factors Contributing to NBA Injuries')
plt.show()


### Feature Selection for Refined Logistic Regression

This cell defines a list of selected features for building a more focused Logistic Regression model, aiming to include only the most relevant variables.

In [ ]:
# Features to include
selected_features = [
    'GamesPlayed',
    'BMI',
    'AvgMins',
    'assists_mean',
    'blocks_mean',
    'steals_mean',
    'fieldGoalsAttempted_mean',
    'threePointersAttempted_mean',
    'freeThrowsAttempted_mean',
    'foulsPersonal_mean',
    'turnovers_mean',
    'total_back_to_backs',
    'avg_rest_days',
    'LeagueTenure',
    'Age',
]

X = df[selected_features]
y = df['Injury']


### Split and Scale Data (Selected Features)

This cell splits the data using the selected features into training and testing sets and then scales these features using `StandardScaler`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### Train Refined Logistic Regression Model

This cell trains a new Logistic Regression model using the previously split and scaled data with the selected features.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)


### Visualize Logistic Regression Coefficients (Refined Model)

This cell displays and visualizes the coefficients of the refined Logistic Regression model, showing the importance of the selected features.

In [ ]:
import pandas as pd

coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', key=abs, ascending=False)

print(coefficients)

# Optional: visualize
plt.figure(figsize=(10,6))
sns.barplot(x='Coefficient', y='Feature', data=coefficients)
plt.title('Feature Importance for Injury Risk')
plt.show()

### Calculate Median Player Features

This cell computes the median values for each of the selected features, creating a 'typical' player profile.

In [ ]:
typical_features = [
    'GamesPlayed',
    'BMI',
    'AvgMins',
    'assists_mean',
    'blocks_mean',
    'steals_mean',
    'fieldGoalsAttempted_mean',
    'threePointersAttempted_mean',
    'freeThrowsAttempted_mean',
    'foulsPersonal_mean',
    'turnovers_mean',
    'total_back_to_backs',
    'avg_rest_days',
    'LeagueTenure',
    'Age',
]

# Compute medians
median_player = df[typical_features].median()

median_player


### Predict Injury Probability for a Median Player

This cell uses the trained Logistic Regression model to predict the probability of injury for a hypothetical 'median' player based on their median feature values.

In [ ]:
# Turn median values into a single-row dataframe
median_df = pd.DataFrame([median_player])

# Scale it using the same scaler as your training data
median_scaled = scaler.transform(median_df)

# Predict probability of injury
prob_injury = model.predict_proba(median_scaled)[0][1]

prob_injury


### Predict Injury Probability for Specific Players

This cell creates a DataFrame for three well-known NBA players, scales their features, and then uses the Logistic Regression model to predict their individual injury probabilities.

In [ ]:
import pandas as pd
# Create DataFrame for the three players
new_players = pd.DataFrame({
    'PlayerFull': ['Giannis Antetokounmpo', 'Payton Pritchard', 'LeBron James'],

    'GamesPlayed': [63, 48, 55],
    'BMI': [24.2, 25.7, 26.8],
    'AvgMins': [31.3, 12.2, 34.9],

    'assists_mean': [5.44, 1.0, 6.18],
    'blocks_mean': [0.76, 0.01, 0.65],
    'steals_mean': [0.8, 0.17, 0.89],

    'fieldGoalsAttempted_mean': [19.54, 3.3, 19.73],
    'threePointersAttempted_mean': [10.67, 1.38, 9.86],
    'freeThrowsAttempted_mean': [11.83, 0.72, 8.06],

    'foulsPersonal_mean': [2.53, 1.1, 1.56],
    'turnovers_mean': [3.81, 0.54, 2.86],

    'total_back_to_backs': [8, 10, 7],
    'avg_rest_days': [3.0, 2.6, 2.92],

    'LeagueTenure': [9+1, 2+1, 19+1],   # add 1
    'Age': [28+1, 25+1, 38+1]           # add 1
})

# List of features for the model
features = [
    'GamesPlayed','BMI','AvgMins','assists_mean','blocks_mean','steals_mean',
    'fieldGoalsAttempted_mean','threePointersAttempted_mean','freeThrowsAttempted_mean','foulsPersonal_mean',
    'turnovers_mean','total_back_to_backs','avg_rest_days','LeagueTenure','Age'
]

# Scale features if your model requires it
new_players_scaled = scaler.transform(new_players[features])

# Predict probabilities
pred_probs = model.predict_proba(new_players_scaled)[:,1]
pred_class = model.predict(new_players_scaled)

# Combine results
new_players['PredictedProbability'] = pred_probs
new_players['PredictedInjury'] = pred_class

# Show predictions
print(new_players[['PlayerFull', 'PredictedProbability','PredictedInjury']])


### Train and Evaluate Random Forest Classifier

This cell prepares the data for a Random Forest Classifier (dropping more columns and handling categorical features), splits it into training and testing sets, trains the model, and then evaluates its performance using a classification report and confusion matrix.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X = df.drop(['Injury', 'PlayerFull', 'PlayerID','InjuryCount','SeasonID','FirstYear'], axis=1)
y = df['Injury']

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


### Visualize Top Feature Importances (Random Forest)

This cell extracts and visualizes the top 20 most important features identified by the Random Forest Classifier, indicating which features contribute most to injury prediction.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:20]  # top 20

plt.figure(figsize=(10,6))
plt.bar(range(len(indices)), importances[indices])
plt.xticks(range(len(indices)), X.columns[indices], rotation=90)
plt.title("Top Feature Importances")
plt.tight_layout()
plt.show()


### Plot Random Forest Confusion Matrix

This cell generates and visualizes the confusion matrix for the Random Forest Classifier, providing a clear overview of the model's true positives, true negatives, false positives, and false negatives.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)
labels = ['No Injury (0)', 'Injury (1)']

# Plot the heatmap
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Random Forest Confusion Matrix')
plt.show()


### Partial Dependence Plots (Random Forest)

This cell generates Partial Dependence Plots for selected features (e.g., `AvgMins`, `total_back_to_backs`) to show the marginal effect of these features on the predicted injury outcome by the Random Forest model.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt

# Example: Partial dependence of AvgMins and B2BTotals
features_to_plot = ['AvgMins', 'total_back_to_backs']
PartialDependenceDisplay.from_estimator(rf, X_train, features_to_plot)
plt.show()


### Display Top Feature Importances (Random Forest)

This cell creates and displays a DataFrame showing the top 20 most important features from the Random Forest model along with their importance scores.

In [ ]:
import pandas as pd
import numpy as np

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

top_features = pd.DataFrame({
    'Feature': X.columns[indices][:20],
    'Importance': importances[indices][:20]
})

print(top_features)


# Creating an interactive dashboard


### Calculate Random Forest Accuracy

This cell calculates and prints the overall accuracy of the Random Forest Classifier on the test set.

In [ ]:
from sklearn.metrics import accuracy_score

# Predict on the test set
y_pred = rf.predict(X_test)

# Compute accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Random Forest Accuracy: {accuracy:.2%}")


## Retrain Random Forest with Selected Features

Train a new Random Forest Classifier using the same `selected_features` that were used for the refined Logistic Regression model. This ensures consistency in input for the dashboard.


In [ ]:
X = df[selected_features]
y = df['Injury']

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Now that the feature matrix X and target variable Y are defined, the next step is to split the data into training and testing sets, initialize the RandomForestClassifier with specified parameters, and then train the model.



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize a RandomForestClassifier
rf_model_selected_features = RandomForestClassifier(n_estimators=300, random_state=42)

# Train the Random Forest Classifier
rf_model_selected_features.fit(X_train, y_train)

print("Random Forest Classifier trained successfully with selected features.")

## Save Models and Scaler

Save both the trained Logistic Regression model, the retrained Random Forest model, and the `StandardScaler` to files. This allows us to load them later for predictions without retraining, which is crucial for a dashboard.


In [ ]:
import joblib

# Save the Logistic Regression model
joblib.dump(model, 'logistic_regression_model.joblib')
print("Logistic Regression model saved as 'logistic_regression_model.joblib'")

# Save the Random Forest model (retrained with selected features)
joblib.dump(rf_model_selected_features, 'random_forest_model.joblib')
print("Random Forest model saved as 'random_forest_model.joblib'")

# Save the StandardScaler
joblib.dump(scaler, 'scaler.joblib')
print("StandardScaler saved as 'scaler.joblib'")

## Create Interactive Input Widgets

Use `ipywidgets` to create interactive input fields (e.g., sliders, text boxes) for each of the `selected_features`. This will form the user input part of the dashboard.


In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, fixed, VBox, HBox, Label, Button, Output
from IPython.display import display

print("ipywidgets imported successfully.")

In [ ]:
feature_widgets = {}

# Define min/max/step for each feature for better interactivity
feature_ranges = {
    'GamesPlayed': {'min': 0, 'max': 82, 'step': 1, 'type': 'int'},
    'BMI': {'min': 18.0, 'max': 35.0, 'step': 0.1, 'type': 'float'},
    'AvgMins': {'min': 0.0, 'max': 48.0, 'step': 0.1, 'type': 'float'},
    'assists_mean': {'min': 0.0, 'max': 15.0, 'step': 0.1, 'type': 'float'},
    'blocks_mean': {'min': 0.0, 'max': 5.0, 'step': 0.01, 'type': 'float'},
    'steals_mean': {'min': 0.0, 'max': 5.0, 'step': 0.01, 'type': 'float'},
    'fieldGoalsAttempted_mean': {'min': 0.0, 'max': 30.0, 'step': 0.1, 'type': 'float'},
    'threePointersAttempted_mean': {'min': 0.0, 'max': 20.0, 'step': 0.1, 'type': 'float'},
    'freeThrowsAttempted_mean': {'min': 0.0, 'max': 20.0, 'step': 0.1, 'type': 'float'},
    'foulsPersonal_mean': {'min': 0.0, 'max': 6.0, 'step': 0.1, 'type': 'float'},
    'turnovers_mean': {'min': 0.0, 'max': 6.0, 'step': 0.1, 'type': 'float'},
    'total_back_to_backs': {'min': 0, 'max': 20, 'step': 1, 'type': 'int'},
    'avg_rest_days': {'min': 0.0, 'max': 10.0, 'step': 0.1, 'type': 'float'},
    'LeagueTenure': {'min': 0, 'max': 20, 'step': 1, 'type': 'int'},
    'Age': {'min': 18, 'max': 45, 'step': 1, 'type': 'int'}
}

# Create widgets for each selected feature
for feature in selected_features:
    if feature in feature_ranges:
        range_info = feature_ranges[feature]
        if range_info['type'] == 'int':
            feature_widgets[feature] = IntSlider(
                min=range_info['min'],
                max=range_info['max'],
                step=range_info['step'],
                description=feature,
                continuous_update=True,
                orientation='horizontal',
                readout=True,
                readout_format='d'
            )
        else:
            feature_widgets[feature] = FloatSlider(
                min=range_info['min'],
                max=range_info['max'],
                step=range_info['step'],
                description=feature,
                continuous_update=True,
                orientation='horizontal',
                readout=True,
                readout_format='.1f'
            )
    else:
        # Fallback for any unlisted features (though all are listed now)
        feature_widgets[feature] = FloatSlider(
            min=df[feature].min(),
            max=df[feature].max(),
            step=(df[feature].max() - df[feature].min()) / 100,
            description=feature,
            continuous_update=True,
            orientation='horizontal',
            readout=True,
            readout_format='.2f'
        )

print(f"Created {len(feature_widgets)} interactive widgets for input.")


Write a Python function that takes the values from the input widgets, converts them into a DataFrame, applies the same scaling used during training (for Logistic Regression), and then uses both the Logistic Regression and Random Forest models to predict the injury probability/class.


In [ ]:
import pandas as pd

def predict_injury(**kwargs):
    # Ensure the order of features matches the training data
    input_data = pd.DataFrame([kwargs], columns=selected_features)

    # --- Logistic Regression Prediction ---
    # Scale the input data using the pre-fitted scaler
    scaled_input_lr = scaler.transform(input_data)
    # Predict probability for Logistic Regression (class 1 - injury)
    lr_prob_injury = model.predict_proba(scaled_input_lr)[0][1]
    # Predict class for Logistic Regression
    lr_predicted_class = model.predict(scaled_input_lr)[0]

    # --- Random Forest Prediction ---
    # Random Forest does not require scaled data for prediction
    # Predict probability for Random Forest (class 1 - injury)
    rf_prob_injury = rf_model_selected_features.predict_proba(input_data)[0][1]
    # Predict class for Random Forest
    rf_predicted_class = rf_model_selected_features.predict(input_data)[0]

    return {
        'LogisticRegression': {
            'Probability': lr_prob_injury,
            'PredictedClass': lr_predicted_class
        },
        'RandomForest': {
            'Probability': rf_prob_injury,
            'PredictedClass': rf_predicted_class
        }
    }

print("Prediction function 'predict_injury' defined.")

## Display Dashboard with Predictions

Combined the input widgets and the prediction function to create an interactive dashboard that updates and displays the predicted injury probability/class for both models based on the user's input.


In [ ]:
output_widget = Output()

def on_predict_button_click(b):
    with output_widget:
        output_widget.clear_output()
        # Collect current values from widgets
        current_features = {name: widget.value for name, widget in feature_widgets.items()}

        # Get predictions
        predictions = predict_injury(**current_features)

        # Display results
        print("--- Prediction Results ---")
        print(f"Logistic Regression Probability: {predictions['LogisticRegression']['Probability']:.4f}")
        print(f"Logistic Regression Predicted Class: {'Injury' if predictions['LogisticRegression']['PredictedClass'] == 1 else 'No Injury'}")
        print(f"\nRandom Forest Probability: {predictions['RandomForest']['Probability']:.4f}")
        print(f"Random Forest Predicted Class: {'Injury' if predictions['RandomForest']['PredictedClass'] == 1 else 'No Injury'}")

print("Prediction output widget and handler function defined.")

To finalize the interactive dashboard, created a predict button, associate its click event with the previously defined handler function, arrange all the feature input widgets, the predict button, and the output display area using `VBox` and `HBox`, and then display the complete layout to the user.



In [ ]:
predict_button = Button(description='Predict Injury')
predict_button.on_click(on_predict_button_click)

# Arrange widgets
input_widgets_list = [widget for widget in feature_widgets.values()]

# Group input widgets into rows for better layout
widget_rows = []
for i in range(0, len(input_widgets_list), 3): # 3 widgets per row
    widget_rows.append(HBox(input_widgets_list[i:i+3]))

# Combine all into a VBox
dashboard_layout = VBox(widget_rows + [predict_button, output_widget])

# Display the dashboard
display(dashboard_layout)

print("Interactive dashboard displayed.")